# 2_ Tuning_and_BestModels

### Step 1: Import libraries and setup paths


In [1]:
import os, re, json
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Optional

In [2]:
# Path
RESULTS_DIR = Path("../results")
SUMMARY_DIR = RESULTS_DIR / "summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

### Step 2: Define a CSV reader
This function safely loads CSV files.
- Returns an empty DataFrame if the file does not exist or fails to load.
- Ensures all expected columns exist, even if missing in the file.


In [3]:
# safe readers
def read_csv_safe(p: Path, cols=None):
    if not p.exists():
        return pd.DataFrame(columns=cols or [])
    try:
        df = pd.read_csv(p)
        if cols is not None:
            for c in cols:
                if c not in df.columns: df[c] = np.nan
            df = df[cols]
        return df
    except Exception:
        return pd.DataFrame(columns=cols or [])

### Step 3: Load CV and ablation results
Reads results from:
- `cv.csv` (5-fold cross-validation)
- `ablation.csv` (preprocessing comparison)

Concatenates them into a single DataFrame `df_cv_all`.  
If columns like `"preproc"` or `"split"` are missing, they are added with default values.


In [4]:
# Load CV results (primary source for picking "best")
cv_cols = ["dataset","model","preproc","split","cv_splits",
           "acc_mean","acc_std","bal_acc_mean","bal_acc_std","f1_macro_mean","f1_macro_std"]

df_cv  = read_csv_safe(RESULTS_DIR / "cv.csv",        cv_cols)
df_abl = read_csv_safe(RESULTS_DIR / "ablation.csv",  cv_cols)
df_cv_all = pd.concat([df_cv, df_abl], ignore_index=True) if len(df_cv) or len(df_abl) else pd.DataFrame(columns=cv_cols)

if "preproc" not in df_cv_all.columns:
    df_cv_all["preproc"] = "baseline"
if "split" not in df_cv_all.columns:
    df_cv_all["split"] = "cv"

### Step 4: Load holdout baseline results
Loads model evaluation results from `*_baseline.csv` files,  
which contain accuracy, balanced accuracy, macro F1, and runtime for each dataset–model pair. Renames columns for consistency and combines all baselines into one DataFrame `df_holdout`.


In [5]:
# Load holdout baselines (fallback if no CV rows exist for a pair) 
# *_baseline.csv: columns expected -> Model, Accuracy, BalancedAcc, MacroF1, Runtime
ho_rows = []

for f in sorted(RESULTS_DIR.glob("*_baseline.csv")):
    m = re.match(r"(.+)_baseline\.csv", f.name)
    if not m: 
        continue
    ds = m.group(1)
    dfb = read_csv_safe(f)
    if not {"Model","Accuracy","BalancedAcc","MacroF1"}.issubset(dfb.columns):
        continue
    tmp = dfb.rename(columns={
        "Model":"model",
        "Accuracy":"acc_holdout",
        "BalancedAcc":"bal_acc_holdout",
        "MacroF1":"f1_macro_holdout",
    })
    tmp["dataset"] = ds
    tmp["preproc"] = "baseline"
    tmp["split"]   = "holdout"
    ho_rows.append(tmp[["dataset","model","preproc","split","acc_holdout","bal_acc_holdout","f1_macro_holdout"]])
    
df_holdout = pd.concat(ho_rows, ignore_index=True) if ho_rows else pd.DataFrame(
    columns=["dataset","model","preproc","split","acc_holdout","bal_acc_holdout","f1_macro_holdout"]
)

### Step 5: Load runtime information
Loads per-model runtime metrics from `runtime.csv`,  
including fit and predict times, and dataset dimensions.  
This will later be merged with accuracy results.


In [6]:
# load runtime 
rt_cols = ["dataset","model","preproc","split","fit_time_sec","pred_time_sec","n_train","n_test","n_features"]
df_rt = read_csv_safe(RESULTS_DIR / "runtime.csv", rt_cols)

### Step 6: Determine best configurations
For each (dataset × model):
- Prefer CV results if available (sorted by macro F1 → accuracy → balanced accuracy)
- Otherwise, fall back to holdout results  
Each selected record is marked with `_chosen_from = "cv"` or `"holdout"`.


In [7]:
# Pick "best" per dataset × model (CV first, else holdout)
best_rows = []
pairs = set()
pairs.update(zip(df_cv_all.get("dataset",[]), df_cv_all.get("model",[])))
pairs.update(zip(df_holdout.get("dataset",[]), df_holdout.get("model",[])))

for ds, mdl in sorted(pairs):
    sub_cv = df_cv_all[(df_cv_all["dataset"]==ds) & (df_cv_all["model"]==mdl)]
    if len(sub_cv):
        # Sort by CV macro-F1 (primary), then accuracy (secondary)
        row = sub_cv.sort_values(["f1_macro_mean","acc_mean","bal_acc_mean"], ascending=False).iloc[0].to_dict()
        row["_chosen_from"] = "cv"
        # Ensure holdout columns exist for uniform schema
        row.update({"f1_macro_holdout":np.nan,"acc_holdout":np.nan,"bal_acc_holdout":np.nan})
    else:
        sub_ho = df_holdout[(df_holdout["dataset"]==ds) & (df_holdout["model"]==mdl)]
        if not len(sub_ho):
            continue
        row = sub_ho.sort_values(["f1_macro_holdout","acc_holdout","bal_acc_holdout"], ascending=False).iloc[0].to_dict()
        # Ensure CV columns exist for uniform schema
        row.update({"cv_splits":np.nan,"f1_macro_mean":np.nan,"acc_mean":np.nan,"bal_acc_mean":np.nan})
        row["_chosen_from"] = "holdout"
    best_rows.append(row)

best = pd.DataFrame(best_rows)

### Step 7: Merge runtime information
If runtime data exist, join them to the “best” result table using (dataset, model, preproc, split) as keys, adding timing and dataset-size metadata.


In [8]:
# Merge runtime if present
if len(df_rt):
    merge_keys = ["dataset","model","preproc","split"]
    best = best.merge(df_rt[merge_keys + ["fit_time_sec","pred_time_sec","n_train","n_test","n_features"]],
                      on=merge_keys, how="left")

### Step 8: Save summarized results
Reorders columns into a consistent schema and fills missing ones with NaN. Exports the final table of selected best results (4 datasets × 3 models) to `summary_best_4x3.csv`, then shows a preview.


In [9]:
# Final column order + save
cols = ["dataset","model","preproc","split","_chosen_from",
        "f1_macro_mean","acc_mean","bal_acc_mean","cv_splits",
        "f1_macro_holdout","acc_holdout","bal_acc_holdout",
        "fit_time_sec","pred_time_sec","n_train","n_test","n_features"]
for c in cols:
    if c not in best.columns: best[c] = np.nan
best = best[cols].sort_values(["dataset","model"])

out_path = SUMMARY_DIR / "summary_best_4x3.csv"
best.to_csv(out_path, index=False)
print("Saved")

best.head(12)

Saved


,dataset,model,preproc,split,_chosen_from,f1_macro_mean,acc_mean,bal_acc_mean,cv_splits,f1_macro_holdout,acc_holdout,bal_acc_holdout,fit_time_sec,pred_time_sec,n_train,n_test,n_features
0,adult,Logistic Regression,baseline,cv,cv,0.773952,0.810786,0.822570,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,adult,Random Forest,baseline,cv,cv,0.793061,0.856485,0.778778,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,adult,SVM (RBF),baseline,cv,cv,0.775923,0.810356,0.830255,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,cancer,Logistic Regression,baseline,cv,cv,0.963050,0.964912,0.964776,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,cancer,Random Forest,baseline,cv,cv,0.951672,0.954386,0.952179,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,cancer,SVM (RBF),baseline,cv,cv,0.970240,0.971930,0.970411,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,loan,Logistic Regression,baseline,cv,cv,0.720340,0.862300,0.733727,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,loan,Random Forest,baseline,cv,cv,0.515114,0.822400,0.516546,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,loan,SVM (RBF),baseline,cv,cv,0.622719,0.790000,0.621334,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,wine,Logistic Regression,baseline,cv,cv,0.212925,0.314917,0.403077,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 9: Enrich summary with all available metrics
Reloads the best summary, then merges:
1. Holdout baseline metrics (accuracy / F1 / runtime)
2. Aggregated runtime statistics (fit / predict times, n_train/test/features)  

Combines duplicates, coalesces missing values, and saves the enriched result to `summary_best_4x3_enriched.csv`.


In [10]:
# Load current best (from memory or file) 
try:
    best
except NameError:
    best = pd.read_csv(SUMMARY_DIR / "summary_best_4x3.csv")

keys = ["dataset","model","preproc"]

# Merge holdout metrics from *_baseline.csv and coalesce
ho_rows = []

for f in sorted(RESULTS_DIR.glob("*_baseline.csv")):
    m = re.match(r"(.+)_baseline\.csv", f.name)
    if not m:
        continue
    ds = m.group(1)
    dfb = pd.read_csv(f)
    if not {"Model","Accuracy","BalancedAcc","MacroF1"}.issubset(dfb.columns):
        continue
    dfb = dfb.rename(columns={
        "Model":"model",
        "Accuracy":"acc_holdout",
        "BalancedAcc":"bal_acc_holdout",
        "MacroF1":"f1_macro_holdout"
    })
    dfb["dataset"] = ds
    dfb["preproc"] = "baseline"
    keep = keys + ["acc_holdout","bal_acc_holdout","f1_macro_holdout"]
    if "Runtime" in dfb.columns:
        dfb = dfb[keep + ["Runtime"]].rename(columns={"Runtime":"total_runtime_sec"})
    else:
        dfb = dfb[keep]
    ho_rows.append(dfb)

if ho_rows:
    df_holdout = pd.concat(ho_rows, ignore_index=True)
    best = best.merge(df_holdout, on=keys, how="left", suffixes=("", "_ho"))
    for c in ["acc_holdout","bal_acc_holdout","f1_macro_holdout","total_runtime_sec"]:
        if c + "_ho" in best.columns:
            best[c] = best[c].combine_first(best[c + "_ho"])
            best.drop(columns=[c + "_ho"], inplace=True)

# Merge runtime.csv (fit/predict etc.) and coalesce
rt_path = RESULTS_DIR / "runtime.csv"

if rt_path.exists():
    df_rt = pd.read_csv(rt_path)
    val_cols = ["fit_time_sec","pred_time_sec","n_train","n_test","n_features"]
    rt_agg = df_rt.groupby(keys, as_index=False)[val_cols].mean()
    best = best.merge(rt_agg, on=keys, how="left", suffixes=("", "_rt"))
    for c in val_cols:
        if c + "_rt" in best.columns:
            best[c] = best[c].combine_first(best[c + "_rt"])
            best.drop(columns=[c + "_rt"], inplace=True)

# Save & preview
out = SUMMARY_DIR / "summary_best_4x3_enriched.csv"
best.to_csv(out, index=False)
print("Saved:", out)
best.sort_values(keys).head(12)

Saved: ..\results\summary\summary_best_4x3_enriched.csv


,dataset,model,preproc,split,_chosen_from,f1_macro_mean,acc_mean,bal_acc_mean,cv_splits,f1_macro_holdout,acc_holdout,bal_acc_holdout,fit_time_sec,pred_time_sec,n_train,n_test,n_features,total_runtime_sec
0,adult,Logistic Regression,baseline,cv,cv,0.773952,0.810786,0.822570,5,0.773677,0.809397,0.825264,0.749603,0.071890,22792.0,9769.0,14.0,0.765087
1,adult,Random Forest,baseline,cv,cv,0.793061,0.856485,0.778778,5,0.793006,0.855973,0.779576,135.469345,1.515574,22792.0,9769.0,14.0,139.428285
2,adult,SVM (RBF),baseline,cv,cv,0.775923,0.810356,0.830255,5,0.770269,0.803153,0.830153,80.417449,12.840235,22792.0,9769.0,14.0,75.818879
3,cancer,Logistic Regression,baseline,cv,cv,0.963050,0.964912,0.964776,5,0.975415,0.976744,0.975415,0.027579,0.009488,199.0,86.0,31.0,0.033768
4,cancer,Random Forest,baseline,cv,cv,0.951672,0.954386,0.952179,5,0.902715,0.906977,0.907376,1.131458,0.062347,199.0,86.0,31.0,1.389289
5,cancer,SVM (RBF),baseline,cv,cv,0.970240,0.971930,0.970411,5,0.926244,0.930233,0.926244,0.025405,0.012088,199.0,86.0,31.0,0.035002
6,loan,Logistic Regression,baseline,cv,cv,0.720340,0.862300,0.733727,5,0.722310,0.853667,0.748771,2.799961,0.061218,7000.0,3000.0,91.0,2.570410
7,loan,Random Forest,baseline,cv,cv,0.515114,0.822400,0.516546,5,0.508087,0.824667,0.512909,22.748267,0.626575,7000.0,3000.0,91.0,22.888345
8,loan,SVM (RBF),baseline,cv,cv,0.622719,0.790000,0.621334,5,0.633096,0.784667,0.621709,18.433564,11.769152,7000.0,3000.0,91.0,18.238285
9,wine,Logistic Regression,baseline,cv,cv,0.212925,0.314917,0.403077,5,0.194905,0.297949,0.274787,0.259702,0.011499,4547.0,1950.0,11.0,0.374234


### Note:
We selected models based on 5-fold CV performance. For reporting, we added holdout metrics as a reference. CV and holdout are largely consistent, except for the wine dataset, indicating sensitivity to small sample size. Runtime measurements confirm SVM with RBF is the most expensive model.